---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！

# 05 - 数据变换（Transforms）

## 学习目标

1. 理解 transforms 的作用和重要性
2. 掌握常用的图像预处理操作
3. 学会使用数据增强技术
4. 理解训练集和验证集的不同 transform 策略

## 1. 什么是 Transform？

Transform（数据变换）是对原始数据进行预处理和增强的函数。

**为什么需要 Transform？**

1. **统一尺寸**：神经网络要求输入尺寸一致
2. **转换格式**：PIL Image → Tensor
3. **数据标准化**：将数据缩放到合适范围，加速训练
4. **数据增强**：增加数据多样性，提高模型泛化能力

**Transform 的使用位置：**

```python
class MyDataset(Dataset):
    def __getitem__(self, index):
        img = Image.open(path)      # PIL Image
        if self.transform:
            img = self.transform(img)  # 在这里应用 transform
        return img, label
```

## 2. Transform 的基本使用

In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import numpy as np

# 创建一个模拟图像
img_array = np.random.randint(0, 255, (100, 100, 3), dtype=np.uint8)
img = Image.fromarray(img_array)

print("原始图像：")
print(f"  类型: {type(img)}")
print(f"  大小: {img.size}")
print(f"  模式: {img.mode}")

# 定义 transform
transform = transforms.ToTensor()

# 应用 transform
img_tensor = transform(img)

print("\n转换后：")
print(f"  类型: {type(img_tensor)}")
print(f"  形状: {img_tensor.shape}")
print(f"  数值范围: [{img_tensor.min():.3f}, {img_tensor.max():.3f}]")

print("\nToTensor 做了什么？")
print("1. PIL Image (H, W, C) → Tensor (C, H, W)")
print("2. [0, 255] → [0, 1]")

## 3. 常用的图像预处理 Transforms

### 3.1 Resize：调整大小

In [ ]:
from torchvision import transforms
from PIL import Image
import numpy as np

# 创建不同大小的图像
img_small = Image.fromarray(np.random.randint(0, 255, (50, 80, 3), dtype=np.uint8))
img_large = Image.fromarray(np.random.randint(0, 255, (200, 300, 3), dtype=np.uint8))

print("原始图像大小：")
print(f"  小图: {img_small.size}")
print(f"  大图: {img_large.size}")

# Resize 到固定大小
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

tensor_small = transform(img_small)
tensor_large = transform(img_large)

print("\nResize 后的形状：")
print(f"  小图: {tensor_small.shape}")
print(f"  大图: {tensor_large.shape}")
print("\n所有图像都被调整到相同大小！")

### 3.2 CenterCrop 和 RandomCrop：裁剪

In [ ]:
img = Image.fromarray(np.random.randint(0, 255, (200, 200, 3), dtype=np.uint8))

# CenterCrop：从中心裁剪
center_crop = transforms.CenterCrop(100)
img_center = center_crop(img)

# RandomCrop：随机位置裁剪
random_crop = transforms.RandomCrop(100)
img_random1 = random_crop(img)
img_random2 = random_crop(img)

print("原始图像大小:", img.size)
print("CenterCrop(100):", img_center.size)
print("RandomCrop(100) 第1次:", img_random1.size)
print("RandomCrop(100) 第2次:", img_random2.size)
print("\nRandomCrop 每次裁剪位置不同（数据增强）")

### 3.3 Normalize：标准化

In [ ]:
# 创建随机 tensor
img_tensor = torch.rand(3, 64, 64)  # [0, 1] 范围

print("标准化前：")
print(f"  均值: {img_tensor.mean(dim=[1, 2])}")
print(f"  标准差: {img_tensor.std(dim=[1, 2])}")

# ImageNet 的标准化参数
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

img_normalized = normalize(img_tensor)

print("\n标准化后：")
print(f"  均值: {img_normalized.mean(dim=[1, 2])}")
print(f"  标准差: {img_normalized.std(dim=[1, 2])}")

print("\nNormalize 公式: output = (input - mean) / std")
print("作用: 将数据缩放到均值为0、标准差为1的分布")

## 4. 数据增强 Transforms

数据增强通过随机变换生成新的训练样本，提高模型泛化能力。

### 4.1 RandomHorizontalFlip：随机水平翻转

In [ ]:
img = Image.fromarray(np.random.randint(0, 255, (100, 100, 3), dtype=np.uint8))

# p=0.5 表示有 50% 的概率翻转
flip = transforms.RandomHorizontalFlip(p=0.5)

print("测试 RandomHorizontalFlip（10次）：")
flip_count = 0
for i in range(10):
    img_flipped = flip(img)
    # 简单判断是否翻转（比较左右两侧）
    if np.array(img_flipped)[:, 0].sum() != np.array(img)[:, 0].sum():
        flip_count += 1
        print(f"  第 {i+1} 次: 翻转了")
    else:
        print(f"  第 {i+1} 次: 没翻转")

print(f"\n翻转次数: {flip_count}/10")
print("每次调用结果可能不同（随机性）")

### 4.2 RandomRotation：随机旋转

In [ ]:
img = Image.fromarray(np.random.randint(0, 255, (100, 100, 3), dtype=np.uint8))

# 随机旋转 -30 到 30 度
rotation = transforms.RandomRotation(degrees=30)

print("RandomRotation（旋转角度随机）：")
for i in range(3):
    img_rotated = rotation(img)
    print(f"  第 {i+1} 次旋转后大小: {img_rotated.size}")

print("\n每次旋转角度不同")

### 4.3 ColorJitter：颜色抖动

In [ ]:
img = Image.fromarray(np.random.randint(100, 200, (100, 100, 3), dtype=np.uint8))

# 随机调整亮度、对比度、饱和度、色调
color_jitter = transforms.ColorJitter(
    brightness=0.5,  # 亮度变化范围 [0.5, 1.5]
    contrast=0.5,    # 对比度变化范围 [0.5, 1.5]
    saturation=0.5,  # 饱和度变化范围 [0.5, 1.5]
    hue=0.1          # 色调变化范围 [-0.1, 0.1]
)

print("ColorJitter 测试：")
transform = transforms.Compose([
    color_jitter,
    transforms.ToTensor()
])

for i in range(3):
    img_jittered = transform(img)
    print(f"  第 {i+1} 次: 均值={img_jittered.mean():.3f}")

print("\n每次颜色都会有所不同")

### 4.4 RandomResizedCrop：随机裁剪并调整大小

In [ ]:
img = Image.fromarray(np.random.randint(0, 255, (200, 200, 3), dtype=np.uint8))

# 随机裁剪并调整到 224x224
transform = transforms.RandomResizedCrop(
    size=224,
    scale=(0.5, 1.0),  # 裁剪面积占原图的 50%-100%
    ratio=(0.75, 1.33)  # 裁剪宽高比范围
)

print("RandomResizedCrop 测试：")
for i in range(3):
    img_cropped = transform(img)
    print(f"  第 {i+1} 次: {img_cropped.size}")

print("\n每次裁剪位置和大小都不同，但输出固定为 224x224")

## 5. 组合多个 Transforms

In [ ]:
from torchvision import transforms

# 使用 Compose 组合多个 transform
transform = transforms.Compose([
    transforms.Resize((256, 256)),           # 1. 调整大小
    transforms.RandomCrop(224),              # 2. 随机裁剪
    transforms.RandomHorizontalFlip(p=0.5),  # 3. 随机翻转
    transforms.ColorJitter(                  # 4. 颜色抖动
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.ToTensor(),                   # 5. 转为 Tensor
    transforms.Normalize(                    # 6. 标准化
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 测试
img = Image.fromarray(np.random.randint(0, 255, (300, 400, 3), dtype=np.uint8))

print("输入图像:", img.size, type(img))
img_transformed = transform(img)
print("输出 Tensor:", img_transformed.shape, type(img_transformed))
print(f"数值范围: [{img_transformed.min():.3f}, {img_transformed.max():.3f}]")

print("\nCompose 按顺序依次应用所有 transform")

## 6. 训练集 vs 验证集的 Transform

### 关键原则

- **训练集**：使用数据增强，增加数据多样性
- **验证集/测试集**：只做基本预处理，不做数据增强

In [ ]:
from torchvision import transforms

# 训练集 transform（包含数据增强）
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),              # 随机裁剪
    transforms.RandomHorizontalFlip(p=0.5),  # 随机翻转
    transforms.RandomRotation(15),           # 随机旋转
    transforms.ColorJitter(                  # 颜色抖动
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 验证集 transform（不做数据增强）
valid_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),    # 中心裁剪（固定）
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("训练集 transform:")
print("  - 使用数据增强（RandomCrop, RandomFlip, ColorJitter 等）")
print("  - 每次获取同一张图片结果都不同")
print()
print("验证集 transform:")
print("  - 只做基本预处理（Resize, CenterCrop）")
print("  - 每次获取同一张图片结果相同")
print()
print("为什么？")
print("  - 训练时需要多样性，帮助模型泛化")
print("  - 验证时需要一致性，便于比较模型性能")

## 7. 实际应用示例

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

class ImageDataset(Dataset):
    def __init__(self, num_samples, transform=None):
        self.num_samples = num_samples
        self.transform = transform
    
    def __getitem__(self, idx):
        # 创建随机图像
        img_array = np.random.randint(0, 255, (200, 200, 3), dtype=np.uint8)
        img = Image.fromarray(img_array)
        
        # 应用 transform
        if self.transform:
            img = self.transform(img)
        
        label = idx % 10
        return img, label
    
    def __len__(self):
        return self.num_samples

# 创建训练集和验证集
train_dataset = ImageDataset(100, transform=train_transform)
valid_dataset = ImageDataset(30, transform=valid_transform)

# 创建 DataLoader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False)

print("训练集：")
for images, labels in train_loader:
    print(f"  批量形状: {images.shape}")
    print(f"  标签形状: {labels.shape}")
    break

print("\n验证集：")
for images, labels in valid_loader:
    print(f"  批量形状: {images.shape}")
    print(f"  标签形状: {labels.shape}")
    break

print("\n完整的数据加载流程：")
print("Dataset → Transform → DataLoader → Batch Tensor")

## 8. 常用 Transform 总结

### 8.1 基础预处理

| Transform | 作用 | 参数 |
|-----------|------|------|
| `Resize(size)` | 调整大小 | size: 目标尺寸 |
| `CenterCrop(size)` | 中心裁剪 | size: 裁剪尺寸 |
| `ToTensor()` | 转为 Tensor | 无 |
| `Normalize(mean, std)` | 标准化 | mean, std: 均值和标准差 |

### 8.2 数据增强

| Transform | 作用 | 参数 |
|-----------|------|------|
| `RandomCrop(size)` | 随机裁剪 | size: 裁剪尺寸 |
| `RandomHorizontalFlip(p)` | 随机水平翻转 | p: 翻转概率 |
| `RandomVerticalFlip(p)` | 随机垂直翻转 | p: 翻转概率 |
| `RandomRotation(degrees)` | 随机旋转 | degrees: 旋转角度范围 |
| `ColorJitter(...)` | 颜色抖动 | brightness, contrast, saturation, hue |
| `RandomResizedCrop(size)` | 随机裁剪+调整 | size: 输出尺寸 |

## 9. 练习题

### 练习 1：设计数据增强策略

为猫狗分类任务设计训练集和验证集的 transform

In [ ]:
# TODO: 设计训练集 transform
# 提示：包含 Resize, RandomCrop, RandomHorizontalFlip, ColorJitter, ToTensor, Normalize

# TODO: 设计验证集 transform
# 提示：只包含 Resize, CenterCrop, ToTensor, Normalize

### 练习 2：观察数据增强的效果

对同一张图像应用多次数据增强，观察结果的差异

In [ ]:
# TODO: 创建一个包含多种数据增强的 transform

# TODO: 对同一张图像应用 5 次，观察每次的均值和标准差

## 10. 高级技巧

### 10.1 自定义 Transform

In [ ]:
class AddGaussianNoise:
    """添加高斯噪声"""
    
    def __init__(self, mean=0.0, std=0.1):
        self.mean = mean
        self.std = std
    
    def __call__(self, tensor):
        """
        参数:
            tensor: 输入的 tensor
        返回:
            添加噪声后的 tensor
        """
        noise = torch.randn(tensor.size()) * self.std + self.mean
        return tensor + noise
    
    def __repr__(self):
        return f"AddGaussianNoise(mean={self.mean}, std={self.std})"

# 使用自定义 transform
transform = transforms.Compose([
    transforms.ToTensor(),
    AddGaussianNoise(mean=0, std=0.1)  # 自定义的 transform
])

# 测试
img = Image.fromarray(np.random.randint(0, 255, (100, 100, 3), dtype=np.uint8))
img_noisy = transform(img)

print("自定义 Transform 测试：")
print(f"输出形状: {img_noisy.shape}")
print(f"数值范围: [{img_noisy.min():.3f}, {img_noisy.max():.3f}]")
print("\n已添加高斯噪声")

### 10.2 Lambda Transform

In [ ]:
# 使用 Lambda 创建简单的自定义 transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x * 2),  # 所有像素值乘以 2
    transforms.Lambda(lambda x: torch.clamp(x, 0, 1))  # 限制在 [0, 1]
])

img = Image.fromarray(np.random.randint(0, 128, (100, 100, 3), dtype=np.uint8))
img_transformed = transform(img)

print("Lambda Transform 测试：")
print(f"输出形状: {img_transformed.shape}")
print(f"数值范围: [{img_transformed.min():.3f}, {img_transformed.max():.3f}]")

## 11. 小结

### 核心要点

1. **Transform 的作用**：
   - 预处理：调整大小、格式转换、标准化
   - 数据增强：增加数据多样性，提高泛化能力

2. **使用原则**：
   - 训练集：使用数据增强
   - 验证集/测试集：只做基本预处理

3. **常用 Transform**：
   - 基础：Resize, ToTensor, Normalize
   - 增强：RandomCrop, RandomHorizontalFlip, ColorJitter

### 标准模板

```python
# 训练集 transform
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# 验证集 transform
valid_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])
```

### 下一步

在下一个 notebook 中，我们将把所有知识整合起来，实现一个**完整的数据加载流程**！